In [7]:
!pip install wbgapi scikit-learn statsmodels matplotlib seaborn numpy pandas

In [8]:
import wbgapi as wb
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, lasso_path, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score

In [9]:
indicators = ['NY.GDP.PCAP.KD.ZG', 'NY.GDP.MKTP.KD.ZG', 'FP.CPI.TOTL.ZG', 'SL.UEM.TOTL.ZS', 'GC.DOD.TOTL.GD.ZS', 'BN.CAB.XOKA.GD.ZS', 'BX.KLT.DINV.WD.GD.ZS', 'NE.TRD.GNFS.ZS', 'NY.GNS.ICTR.ZS', 'FM.LBL.BMNY.ZG', 'FR.INR.LEND', 'FR.INR.RINR', 'DT.DOD.DECT.GN.ZS', 'FI.RES.TOTL.MO', 'FS.AST.PRVT.GD.ZS', 'NE.EXP.GNFS.ZS', 'NE.IMP.GNFS.ZS', 'SP.DYN.LE00.IN', 'SP.POP.GROW', 'SL.TLF.CACT.ZS', 'NE.GDI.TOTL.ZS', 'NE.CON.GOVT.ZS', 'MS.MIL.XPND.GD.ZS', 'TX.VAL.TECH.MF.ZS', 'NV.IND.MANF.ZS', 'NV.AGR.TOTL.ZS']

df = wb.data.DataFrame(indicators, time=range(2013, 2020)).mean(axis=1).unstack()
df = df.dropna(thresh=int(0.6*df.shape[1])).dropna(axis=1, thresh=int(0.6*df.shape[0])).fillna(df.median())

df['crisis'] = (df['NY.GDP.PCAP.KD.ZG'] < 0).astype(int)
X, y = df.drop(columns=['NY.GDP.PCAP.KD.ZG', 'crisis']), df['crisis']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X.columns)

print(f"Dimensions: {X.shape}")
print(f"Crisis countries: {y.sum()}")
print(f"Noncrisis countries: {len(y) - y.sum()}")
print(f"Base rate: {y.mean():.2%}")

Dimensions: (235, 21)
Crisis countries: 38
Noncrisis countries: 197
Base rate: 16.17%


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['NY.GDP.PCAP.KD.ZG', 'crisis']), df['NY.GDP.PCAP.KD.ZG'], test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ols = LinearRegression().fit(X_train_scaled, y_train)

train_r2 = ols.score(X_train_scaled, y_train)
test_r2 = ols.score(X_test_scaled, y_test)
r2_difference = train_r2 - test_r2
pn_ratio = X_train_scaled.shape[1] / X_train_scaled.shape[0]

print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")
print(f"Gap: {r2_difference}")
print(f"pn ratio: {pn_ratio}")

# When there are to many predictors compared to the amount of data, the model
# uses up all of its degrees of freedom to memorize the training set. This causes
# the model to have a low bias, causing the training r2 to inflate. It also suffers
# from high variance when being tested on new data, causing a drop off in the gap.

Train R2: 0.9987284780639999
Test R2: 0.9972010252337591
Gap: 0.0015274528302408052
pn ratio: 0.12804878048780488


In [ ]:
from sklearn.metrics import mean_squared_error


ridge = RidgeCV(cv=5).fit(X_train_scaled, y_train)
ridge_alpha = ridge.alpha_
ridge_nonzero = sum(ridge.coef_ != 0)
ridge_train_r2 = ridge.score(X_train_scaled, y_train)
ridge_test_r2 = ridge.score(X_test_scaled, y_test)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_scaled)))

lasso = LassoCV(cv=5).fit(X_train_scaled, y_train)
lasso_alpha = lasso.alpha_
lasso_nonzero = sum(lasso.coef_ != 0)
lasso_train_r2 = lasso.score(X_train_scaled, y_train)
lasso_test_r2 = lasso.score(X_test_scaled, y_test)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_scaled)))

ols_rmse = np.sqrt(mean_squared_error(y_test, ols.predict(X_test_scaled)))
ols_nonzero = sum(ols.coef_ != 0)